# G-LEAP Inference Tutorial

This notebook shows how to:
1) Set up the environment
2) Prepare or load graph dictionaries
3) Run CV ensemble inference
4) Inspect results

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomoyaS19/gleap/blob/main/examples/tutorial_inference.ipynb)

## Setup

First, install the required dependencies and clone the repository (if running on Colab).

In [ ]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repository
    !git clone https://github.com/TomoyaS19/gleap.git
    %cd gleap
    
    # Install dependencies
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    !pip install -q torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
    !pip install -q esm rdkit unimol_tools biopython
    
    print("Setup complete!")
else:
    print("Running locally - ensure environment is activated")

## Load sample data

Each graph dict is stored as `.npy` with format:
- Protein: `{UniProt_ID.pdb: (num_nodes, features, edge_index)}`
- Ligand: `{smiles: (num_nodes, features, edge_index)}`

In [ ]:
import os
from pathlib import Path

# Set up paths
if IN_COLAB:
    repo_root = Path("/content/gleap")
else:
    # Find repo root from current directory
    def find_repo_root(start: Path) -> Path:
        for path in [start] + list(start.parents):
            if (path / "src" / "inference_cv.py").exists():
                return path
        raise FileNotFoundError("Could not find repo root")
    repo_root = find_repo_root(Path.cwd())

# Change to repo root for relative paths
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

# Verify required files
required = ["src/inference_cv.py", "models/drug_screening", 
            "examples/input_sample.csv", "data/protein_graph_sample.npy"]
for f in required:
    print(f"{f}: {'OK' if Path(f).exists() else 'MISSING'}")

In [ ]:
import numpy as np

protein_graph = np.load("data/protein_graph_sample.npy", allow_pickle=True).item()
ligand_graph = np.load("data/ligand_graph_sample.npy", allow_pickle=True).item()

print(f"Proteins: {len(protein_graph)}")
print(f"Ligands:  {len(ligand_graph)}")

# Check feature dimensions
p_key, (p_n, p_feat, p_edge) = next(iter(protein_graph.items()))
l_key, (l_n, l_feat, l_edge) = next(iter(ligand_graph.items()))

print(f"Protein: {p_key}, nodes={p_n}, feat_dim={len(p_feat[0])}")
print(f"Ligand: nodes={l_n}, feat_dim={len(l_feat[0])}")

In [ ]:
import pandas as pd

df = pd.read_csv("examples/input_sample.csv")
print(f"Input samples: {len(df)}")
print(df[["UniProt ID", "smiles"]].head())

## (Optional) Build your own graph dictionaries

This section demonstrates how to build graph dictionaries from raw data.
We'll download a sample PDB from AlphaFold DB and use the included sample SMILES.

**Note:** AlphaFold DB structures are available under CC-BY 4.0 license.

In [ ]:
import requests
from pathlib import Path

pdb_dir = Path("data/pdb_sample")
pdb_dir.mkdir(exist_ok=True)

# GPCRs to download from AlphaFold DB
uniprot_ids = ["P42866", "P25103", "P21462", "Q92633", "P47211"]

def download_alphafold_pdb(uniprot_id, output_dir):
    """Download PDB from AlphaFold DB."""
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    response = requests.get(api_url)
    if response.status_code != 200:
        raise ValueError(f"API error for {uniprot_id}")

    pdb_url = response.json()[0]["pdbUrl"]
    pdb_response = requests.get(pdb_url)

    output_path = output_dir / f"{uniprot_id}.pdb"
    output_path.write_text(pdb_response.text)
    print(f"Downloaded: {uniprot_id}")
    return output_path

# Download PDBs
for uid in uniprot_ids:
    pdb_path = pdb_dir / f"{uid}.pdb"
    if not pdb_path.exists():
        download_alphafold_pdb(uid, pdb_dir)
    else:
        print(f"Already exists: {uid}.pdb")

In [ ]:
# Build protein graphs (requires GPU)
!python -u scripts/build_protein_graphs_esmc.py \
    --pdb_dir data/pdb_sample \
    --output_graph data/protein_graph_tutorial.npy \
    --output_embeddings data/protein_embeddings_tutorial.pt

In [ ]:
# Build ligand graphs (requires GPU)
!python -u scripts/build_compound_graphs_unimol.py \
    --csv_file examples/ligands_sample.csv \
    --output_graph data/ligand_graph_tutorial.npy \
    --use_gpu

## Run CV ensemble inference (sample data)

This uses the published 10-fold ensemble with the sample graphs.

In [ ]:
# Run inference with 10-fold CV ensemble
!python -u src/inference_cv.py \
    --model_dir models/drug_screening \
    --csv_file examples/input_sample.csv \
    --protein_graph data/protein_graph_sample.npy \
    --ligand_graph data/ligand_graph_sample.npy \
    --output_file examples/output.csv

In [ ]:
# View results
df_out = pd.read_csv("examples/output.csv")
print(f"Predictions: {len(df_out)}")
print(f"Positive (score > 0.5): {sum(df_out['CV_Ensemble_Binary'])}")
df_out[["UniProt_ID", "SMILES", "CV_Ensemble_Score", "CV_Ensemble_Binary", "Label"]].head(10)

## Metabolite screening (optional)

```bash
!python -u src/inference_cv.py \
    --model_dir models/metabolite_screening \
    --csv_file examples/metabolite_input_sample.csv \
    --protein_graph data/metabolite_protein_graph_sample.npy \
    --ligand_graph data/metabolite_ligand_graph_sample.npy \
    --output_file examples/metabolite_output.csv
```